# 1. По тональности, как в статье Financial sentiment analysis using FinBERT

In [ ]:
!pip install tqdm

from tqdm import tqdm


In [ ]:
# ===============================
# 0. Библиотеки
# ===============================
!pip install transformers torch pandas scikit-learn

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
# ===============================
# 1. FinBERT сентимент
# ===============================
finbert_model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)
finbert = AutoModelForSequenceClassification.from_pretrained(finbert_model_name)

def get_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = finbert(**inputs)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
    sentiment_score = -1*probs[0,0] + 0*probs[0,1] + 1*probs[0,2]
    return sentiment_score.item()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-tone
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ===============================
# 2. Подготовка датасета
# ===============================
class StockDataset(Dataset):
    def __init__(self, news_csv, stock_csv, seq_len=5):
        """
        news_csv: CSV с колонками 'date', 'title', 'sentiment' (предвычисленный)
        stock_csv: CSV с колонкой 'close'
        seq_len: длина LSTM последовательности
        """
        # --- Новости ---
        news = pd.read_csv(news_csv, parse_dates=['date'])
        news = news.sort_values('date')
        # Используем уже готовый сентимент
        self.daily_sentiment = news.groupby('date')['sentiment'].mean().reset_index()

        # --- Котировки ---
        stock = pd.read_csv(stock_csv, parse_dates=['begin'])
        stock = stock.sort_values('begin')
        self.stock_prices = stock[['begin','close']].rename(columns={'begin':'date','close':'Close'})

        # --- Объединяем сентимент и котировки ---
        self.data = pd.merge(self.stock_prices, self.daily_sentiment,
                             left_on='date', right_on='date', how='left')
        # Если новостей нет в этот день, ставим сентимент = 0
        self.data['sentiment'].fillna(0, inplace=True)

        # --- Нормализация цены ---
        self.scaler = MinMaxScaler()
        self.data['Close_scaled'] = self.scaler.fit_transform(self.data['Close'].values.reshape(-1,1))

        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        # Последовательность цен и сентиментов
        seq_prices = self.data['Close_scaled'].values[idx:idx+self.seq_len]
        seq_sentiment = self.data['sentiment'].values[idx:idx+self.seq_len]
        X = np.column_stack([seq_prices, seq_sentiment])
        y = self.data['Close_scaled'].values[idx+self.seq_len]
        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [ ]:
# ===============================
# 3. LSTM модель
# ===============================
class LSTMModel(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=100, num_layers=2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc1 = nn.Linear(hidden_dim, 25)
        self.fc2 = nn.Linear(25,1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # берём последний output
        out = self.fc1(out)
        out = self.fc2(out)
        return out


In [ ]:
# ===============================
# 4. Функция тренировки
# ===============================
def train_model(model, dataset, epochs=5, batch_size=32, lr=1e-3):
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X, y in train_loader:
            optimizer.zero_grad()
            output = model(X)
            loss = criterion(output.flatten(), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X, y in val_loader:
                output = model(X)
                loss = criterion(output.flatten(), y)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        print(f"Epoch {epoch+1}: Train Loss={train_loss:.6f}, Val Loss={val_loss:.6f}")


In [ ]:
# --- Загружаем часовые котировки Сбера и фильтруем ---
# prices = pd.read_csv("/content/drive/MyDrive/ВКР/data/stocks/sber_1h_2024_2026.csv", parse_dates=['begin'])
# prices_2025 = prices[prices['begin'] >= '2025-01-01']
# prices_2025.to_csv("/content/drive/MyDrive/ВКР/data/stocks/sber_prices_small.csv", index=False)

# также делаем с новостями
news = pd.read_csv("/content/drive/MyDrive/ВКР/data/all_news.csv", parse_dates=['date'])
news_2025 = news[news['date'] >= '2025-01-01']
news_2025.to_csv("/content/drive/MyDrive/ВКР/data/all_news_2025.csv", index=False)


In [ ]:
# --- Загружаем новости ---
news_csv = "/content/drive/MyDrive/ВКР/data/all_news_2025.csv"
news = pd.read_csv(news_csv, parse_dates=['date'])
news = news.sort_values('date')

# --- Загружаем FinBERT ---
finbert_model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)
finbert = AutoModelForSequenceClassification.from_pretrained(finbert_model_name)

# --- Функция сентимента ---
def get_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = finbert(**inputs)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
    sentiment_score = -1*probs[0,0] + 0*probs[0,1] + 1*probs[0,2]
    return sentiment_score.item()

# --- Прогоняем все новости один раз ---
news['sentiment'] = [get_sentiment(t) for t in tqdm(news['title'], desc="FinBERT Sentiment")]

# --- Сохраняем CSV с уже готовым сентиментом ---
sentiment_csv = "/content/drive/MyDrive/ВКР/data/all_news_2025_sentiment.csv"
news.to_csv(sentiment_csv, index=False)

print("Сентимент добавлен, CSV сохранён:", sentiment_csv)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-tone
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
FinBERT Sentiment: 100%|██████████| 38954/38954 [1:11:37<00:00,  9.06it/s]


Сентимент добавлен, CSV сохранён: /content/drive/MyDrive/ВКР/data/all_news_2025_sentiment.csv


In [ ]:
# ===============================
# 5. Пример использования
# ===============================
news_csv = "/content/drive/MyDrive/ВКР/data/all_news_2025_sentiment.csv"# все новости по сберу с 2025
stock_csv = "/content/drive/MyDrive/ВКР/data/stocks/sber_prices_small.csv" # все почасовые котировки сбера с 2025
dataset = StockDataset(news_csv, stock_csv, seq_len=5)
model = LSTMModel(input_dim=2)
train_model(model, dataset, epochs=5)

/tmp/ipython-input-1577124098.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.data['sentiment'].fillna(0, inplace=True)


Epoch 1: Train Loss=0.022218, Val Loss=0.001143
Epoch 2: Train Loss=0.000826, Val Loss=0.001091
Epoch 3: Train Loss=0.000797, Val Loss=0.001091
Epoch 4: Train Loss=0.000765, Val Loss=0.001130
Epoch 5: Train Loss=0.000752, Val Loss=0.001094


!!! надо потюнить seq_len, hidden_dim, num_layers

# 2. По статье News to Numbers NLP Stock Return Predictions

In [ ]:
# ===============================
# 0. Библиотеки
# ===============================
!pip install transformers torch pandas scikit-learn tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

In [ ]:
# ===============================
# 1. FinBERT сентимент (эмбеддинги заранее)
# ===============================
finbert_model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)
finbert = AutoModelForSequenceClassification.from_pretrained(finbert_model_name)

def get_sentiment(text):
    """Возвращает сентимент score от -1 до 1"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = finbert(**inputs)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
    sentiment_score = -1*probs[0,0] + 0*probs[0,1] + 1*probs[0,2]
    return sentiment_score.item()

def build_news_sentiment(news_csv):
    """Считает сентимент заранее для всех новостей"""
    news = pd.read_csv(news_csv, parse_dates=['date'])
    news = news.sort_values('date')

    sentiments = []
    for text in tqdm(news['title'], desc="FinBERT sentiment"):
        sentiments.append(get_sentiment(text))
    news['sentiment'] = sentiments

    # Усредняем по часу
    news['hour'] = news['date'].dt.floor('H')
    daily_sentiment = news.groupby('hour')['sentiment'].mean().reset_index()
    return daily_sentiment

In [ ]:
# ===============================
# 2. Датасет LSTM
# ===============================
class StockDataset(Dataset):
    def __init__(self, stock_csv, index_csv, news_sentiment, seq_len=5):
        """
        stock_csv: котировки акции
        index_csv: котировки индекса (MOEX)
        news_sentiment: DataFrame с колонками 'hour' и 'sentiment'
        """
        # --- Котировки акции ---
        stock = pd.read_csv(stock_csv, parse_dates=['begin'])
        stock = stock.sort_values('begin')
        stock['log_return'] = np.log(stock['close']/stock['close'].shift(1))
        stock = stock.dropna()

        # --- Котировки индекса ---
        index = pd.read_csv(index_csv, parse_dates=['begin'])
        index = index.sort_values('begin')
        index['log_return'] = np.log(index['close']/index['close'].shift(1))
        index = index.dropna()

        # --- Объединяем акции, индекс, новости ---
        data = pd.merge(stock[['begin','log_return']], index[['begin','log_return']],
                        left_on='begin', right_on='begin', how='left', suffixes=('_stock','_index'))
        data = pd.merge(data, news_sentiment, left_on='begin', right_on='hour', how='left')
        data['sentiment'].fillna(0, inplace=True)

        # --- Целевая переменная: относительная доходность ---
        data['target'] = data['log_return_stock'] - data['log_return_index']

        # --- Нормализация ---
        scaler = MinMaxScaler()
        data[['log_return_stock','log_return_index','sentiment']] = scaler.fit_transform(
            data[['log_return_stock','log_return_index','sentiment']])

        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        seq = self.data.iloc[idx:idx+self.seq_len]
        X = seq[['log_return_stock','log_return_index','sentiment']].values
        y = self.data['target'].iloc[idx+self.seq_len]
        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [ ]:
# ===============================
# 3. LSTM модель
# ===============================
class LSTMModel(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=100, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc1 = nn.Linear(hidden_dim, 25)
        self.fc2 = nn.Linear(25,1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # последний output
        out = self.fc1(out)
        out = self.fc2(out)
        return out

In [ ]:
# ===============================
# 4. Функция тренировки
# ===============================
def train_model(model, dataset, epochs=5, batch_size=32, lr=1e-3):
    train_size = int(0.9*len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X, y in train_loader:
            optimizer.zero_grad()
            output = model(X)
            loss = criterion(output.flatten(), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X, y in val_loader:
                output = model(X)
                loss = criterion(output.flatten(), y)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        print(f"Epoch {epoch+1}: Train Loss={train_loss:.6f}, Val Loss={val_loss:.6f}")


In [ ]:
# ===============================
# 5. Пример использования
# ===============================
# 1. Считаем сентимент для всех новостей заранее
# news_sentiment = build_news_sentiment("/content/drive/MyDrive/ВКР/data/all_news_2025.csv")

# # 2. Загружаем датасет LSTM
# dataset = StockDataset(
#     stock_csv="/content/drive/MyDrive/ВКР/data/stocks/sber_prices_small.csv",
#     index_csv="/content/drive/MyDrive/ВКР/data/stocks/moex_index_1h.csv",
#     news_sentiment=news_sentiment,
#     seq_len=5
# )

# # 3. Создаем модель
# model = LSTMModel(input_dim=3)

# # 4. Тренируем
# train_model(model, dataset, epochs=5, batch_size=32)

# также потом нужно:

- добавить относительное изменение цены как целевую переменную
- эксперименты с подаваемыми данными: только заголовки/описания/полные тексты или какие-то комбинации
- эксперименты с гиперпараметрами: окно прогноза, время между прогнозами (1d, 1h, 10m), сколько слоев и нейронов в LSTM

Для этого нужно:
- попарсить больше данных котировок (а также индекс, золото)
- попарсить больше данных новостей (получить полные тексты, описания, разобраться с датами)